# Librerías

In [1]:
import pandas as pd
import numpy as np
import re
import pathlib as Path
import unicodedata

# Documents

Extraer los datos de los últimos 5 años (2020 - 2024):
Hojas de formación en las empresas, contexto y errores muestreo
Extraemos todos los datos respecto a formación en las empresas pero después analizaremos solo (en el caso de algunas tablas de formación cogeremos ya las específicas de servicios [e.g. 2024 EAL-23 y 23c]):
CCAA: Total y Cataluña y otras CCAA donde tenemos sede (Madrid, Valencia, Sevilla y Bilbao)
Tamaño empresa: más de 499
Sector Transporte y almacenamiento y cuando agregado servicios

TABLAS CONTEXTO
2020, 2021, 2022, 2023, 2024:
EAL-C1, EAL-C2, EAL-C3

TABLAS FORMACIÓN EMPRESAS
2020, 2021, 2022:
EAL-16, EAL-17, EAL-18, EAL-19, EAL-20, EAL-21, EAL-22, EAL-24, EAL-25

2023, 2024 añadido:
EAL-18a, EAL-18b, EAL-18c

ERRORES MUESTREO
2020, 2021, 2022, 2023, 2024:
EAL-M1


He hagut de fer: pip install openpyxl (és l'engine de pandas per excel més nous)

In [2]:

from pathlib import Path


def encontrar_raiz_repo(inicio=None):
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")


REPO_ROOT = encontrar_raiz_repo()
INTERIM_DIR = REPO_ROOT / "Equip_31/Data/interim/EAL/2024"
ANIO = 2024

hojas_formacion_fede = [
    "EAL-16", "EAL-17", "EAL-18", "EAL-18a", "EAL-18b", "EAL-18c", "EAL-19", "EAL-20"
]

print("Raíz repo:", REPO_ROOT)
print("Carpeta interim:", INTERIM_DIR.relative_to(REPO_ROOT))
print("Carpeta existe:", INTERIM_DIR.exists())

faltantes = [hoja for hoja in hojas_formacion_fede if not (INTERIM_DIR / f"{hoja}.csv").exists()]
print("Archivos faltantes:", faltantes if faltantes else "ninguno")


Raíz repo: /Users/fedeur/ProjecteData
Carpeta interim: Equip_31/Data/interim/EAL/2024
Carpeta existe: True
Archivos faltantes: ninguno


In [3]:

# Vista rápida de los CSV disponibles en interim, sin crear archivos nuevos.
archivos_interim = sorted(INTERIM_DIR.glob("*.csv"))
for ruta in archivos_interim:
    print(ruta.name)


EAL-16.csv
EAL-17.csv
EAL-18.csv
EAL-18a.csv
EAL-18b.csv
EAL-18c.csv
EAL-19.csv
EAL-20.csv
EAL-21.csv
EAL-22.csv
EAL-23.csv
EAL-24.csv
EAL-25.csv
EAL-C1.csv
EAL-C2.csv
EAL-C3.csv
EAL-M1.csv


In [4]:

# Comprobación estructural mínima de los CSV ya existentes en interim.
resumen_interim = []
for ruta in archivos_interim:
    df_tmp = pd.read_csv(ruta, header=None, dtype=str, keep_default_na=False)
    resumen_interim.append({
        "archivo": ruta.name,
        "filas": df_tmp.shape[0],
        "columnas": df_tmp.shape[1],
    })

pd.DataFrame(resumen_interim)


,archivo,filas,columnas
0,EAL-16.csv,93,6
1,EAL-17.csv,42,7
2,EAL-18.csv,43,7
3,EAL-18a.csv,41,7
4,EAL-18b.csv,42,9
5,EAL-18c.csv,41,2
6,EAL-19.csv,11,5
7,EAL-20.csv,17,7
8,EAL-21.csv,17,5
9,EAL-22.csv,29,10


## Normalización estructural de EAL-16 a EAL-20

A partir de aquí se leen los CSV ya extraídos en `Equip_31/Data/interim/EAL/2024`. La salida se alinea con el estilo de Sabina: columnas de metadatos en minúscula (`anio`, `tabla`, `pregunta`, `ambito`) y variables de respuesta en columnas limpias.


In [5]:

# ============================================================
# CONFIGURACIÓN
# ============================================================

import re
import unicodedata
import pandas as pd
from pathlib import Path

ANIO = 2024
INTERIM_DIR_REL = Path("Equip_31/Data/interim/EAL/2024")
INTERIM_DIR = REPO_ROOT / INTERIM_DIR_REL

print("Ruta relativa:", INTERIM_DIR_REL)
print("Carpeta existe:", INTERIM_DIR.exists())

if not INTERIM_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta interim esperada: {INTERIM_DIR}")


Ruta relativa: Equip_31/Data/interim/EAL/2024
Carpeta existe: True


## Funciones auxiliares

Estas funciones leen las hojas desde `interim`, limpian espacios y generan nombres de columnas homogéneos para que los dataframes queden comparables con los de Sabina.


In [6]:

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def limpiar_texto(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).replace("\n", " ")
    valor = re.sub(r"\s+", " ", valor)
    return valor.strip()


def normalizar_columna(valor):
    valor = limpiar_texto(valor).upper()
    valor = re.sub(r"\s+", " ", valor)
    return valor


def leer_csv_eal(hoja):
    ruta = INTERIM_DIR / f"{hoja}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el CSV esperado: {ruta}")

    df = pd.read_csv(ruta, header=None, dtype=str, keep_default_na=False)
    df = df.map(limpiar_texto)
    df = df.replace("", pd.NA)
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.fillna("").reset_index(drop=True)
    return df


def extraer_tabla_pregunta(texto):
    texto = limpiar_texto(texto)
    match = re.match(r"^(EAL-\d+[A-Za-z]?)\.\s*(.*)$", texto)
    if not match:
        return "", texto
    return match.group(1), match.group(2).strip()


def detectar_titulo(df, hoja):
    for valor in df.iloc[:, 0].tolist():
        texto = limpiar_texto(valor)
        if texto.upper().startswith(f"{hoja.upper()}."):
            return texto
    return ""


def metadatos_base(anio, tabla, pregunta, ambito):
    return {
        "anio": anio,
        "tabla": tabla,
        "pregunta": pregunta,
        "ambito": ambito,
    }


def hacer_columnas_unicas(columnas):
    resultado = []
    contador = {}
    for columna in columnas:
        columna = normalizar_columna(columna) or "SIN_TITULO"
        if columna not in contador:
            contador[columna] = 1
            resultado.append(columna)
        else:
            contador[columna] += 1
            resultado.append(f"{columna}_{contador[columna]}")
    return resultado


def to_long_format(df, id_vars, value_vars, var_name, value_name="porcentaje"):
    df_long = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name=var_name,
        value_name=value_name,
    )
    return df_long.dropna(subset=[value_name]).reset_index(drop=True)


## Lectura del CSV original

Se muestra EAL-16 tal como quedó extraído en `interim`, para poder contrastarlo visualmente con el Excel raw cuando haga falta.


In [7]:

# ============================================================
# LECTURA DEL CSV ORIGINAL
# ============================================================

df_16_raw = leer_csv_eal("EAL-16")

print("Dimensión EAL-16 raw:", df_16_raw.shape)
display(df_16_raw.head(20))


Dimensión EAL-16 raw: (93, 6)


,0,1,2,3,4,5
0,ENCUESTA ANUAL LABORAL,,,,,EAL
1,,,,,,Volver al índice
2,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,,,,,
3,Año 2024. Porcentaje sobre el total de empresas.,,,,,
4,,TOTAL,NADA,POCO,BASTANTE,MUCHO
5,De dirección,100,10.414,18.818,40.283,30.484
6,De trabajo en equipo,100,2.224,5.822,45.125,46.829
7,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336
8,Administrativas de oficina,100,11.117,25.313,37.788,25.782
9,De resolución de problemas (localización de pr...,100,5.893,15.752,44.063,34.292


## EAL-16 en formato ancho

Una fila por competencia y bloque de la hoja. Esta tabla queda con el mismo estilo base de Sabina: `anio`, `tabla`, `pregunta`, `ambito` y respuestas (`TOTAL`, `NADA`, `POCO`, `BASTANTE`, `MUCHO`).


In [8]:

# ============================================================
# EAL-16 A FORMATO ANCHO
# ============================================================

COLUMNAS_GRADO = ["TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"]

def fila_es_cabecera_grado(row):
    valores = [normalizar_columna(x) for x in row.tolist()]
    return set(COLUMNAS_GRADO).issubset(set(valores))


def process_eal16_wide(df, anio=2024):
    registros = []
    tabla_actual = None
    pregunta_actual = None
    columnas_actuales = None

    for _, row in df.iterrows():
        primera_celda = limpiar_texto(row.iloc[0])
        tabla_detectada, pregunta_detectada = extraer_tabla_pregunta(primera_celda)

        if re.match(r"^EAL-16[A-Za-z]?$", tabla_detectada):
            tabla_actual = tabla_detectada
            pregunta_actual = pregunta_detectada
            columnas_actuales = None
            continue

        if tabla_actual and fila_es_cabecera_grado(row):
            columnas_actuales = [normalizar_columna(x) for x in row.tolist()]
            continue

        if not (tabla_actual and columnas_actuales):
            continue

        ambito = primera_celda
        if ambito == "":
            continue

        registro = metadatos_base(anio, tabla_actual, pregunta_actual, ambito)
        for posicion, columna in enumerate(columnas_actuales):
            if columna in COLUMNAS_GRADO:
                registro[columna] = pd.to_numeric(limpiar_texto(row.iloc[posicion]), errors="coerce")
        registros.append(registro)

    columnas = ["anio", "tabla", "pregunta", "ambito", *COLUMNAS_GRADO]
    return pd.DataFrame(registros)[columnas]


df_16 = process_eal16_wide(df_16_raw, anio=ANIO)

print("df_16:", df_16.shape)
print("Esperado: 7 bloques x 10 categorías = 70 registros")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_16)


df_16: (70, 9)
Esperado: 7 bloques x 10 categorías = 70 registros


,anio,tabla,pregunta,ambito,TOTAL,NADA,POCO,BASTANTE,MUCHO
0,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,De dirección,100,10.414,18.818,40.283,30.484
1,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,De trabajo en equipo,100,2.224,5.822,45.125,46.829
2,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336
3,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,Administrativas de oficina,100,11.117,25.313,37.788,25.782
4,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,"De resolución de problemas (localización de problemas o fallos, análisis de sus causas y búsqueda de soluciones)",100,5.893,15.752,44.063,34.292
5,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,En lenguas extranjeras,100,23.022,40.356,24.568,12.054
6,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,Básicas de cálculo y/o comunicación oral o escrita,100,22.688,36.513,29.697,11.102
7,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,Generales de tecnologías de la información,100,13.593,29.346,39.173,17.888
8,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,Profesionales de tecnologías de la información,100,26.908,37.656,24.493,10.944
9,2024,EAL-16,EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS,"Competencias técnicas, prácticas y otras específicas del puesto de trabajo",100,13.417,18.861,36.496,31.226


## EAL-17 y EAL-18 en formato ancho

Estas hojas tienen cabeceras agrupadas. Se conservan en ancho para que la lectura sea directa y se usan los mismos metadatos base: `anio`, `tabla`, `pregunta`, `ambito`.


In [9]:

# ============================================================
# FUNCIONES PARA EAL-17 Y EAL-18 EN FORMATO ANCHO
# ============================================================

SECCIONES_FILA = {
    "TAMAÑO DE LA EMPRESA": "tamano_empresa",
    "ACTIVIDAD ECONÓMICA": "sector",
    "COMUNIDAD AUTÓNOMA": "ccaa",
}


def nombre_ambito(categoria, seccion_actual):
    if categoria.upper() == "TOTAL":
        return "total_empresas"
    return seccion_actual or "total_empresas"


def columnas_agrupadas(df, fila_grupo, fila_subgrupo, primera_columna_valor=1):
    grupo_superior = [limpiar_texto(x) for x in df.iloc[fila_grupo].tolist()]
    subvariable = [limpiar_texto(x) for x in df.iloc[fila_subgrupo].tolist()]

    columnas = {}
    grupo_actual = ""
    for col in range(primera_columna_valor, df.shape[1]):
        if grupo_superior[col]:
            grupo_actual = grupo_superior[col]
        sub = subvariable[col]

        if col == primera_columna_valor and not sub:
            nombre = "TOTAL"
        elif grupo_actual and sub:
            nombre = f"{grupo_actual} - {sub}"
        else:
            nombre = grupo_actual or sub or "TOTAL"

        columnas[col] = normalizar_columna(nombre)
    return columnas


def process_eal17_18_wide(hoja, anio=2024):
    df = leer_csv_eal(hoja)
    titulo = detectar_titulo(df, hoja)
    tabla, pregunta = extraer_tabla_pregunta(titulo)
    columnas_valor = columnas_agrupadas(df, fila_grupo=4, fila_subgrupo=5)

    registros = []
    seccion_actual = None
    for _, row in df.iloc[6:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "" or categoria.startswith("("):
            continue

        categoria_norm = categoria.upper()
        if categoria_norm in SECCIONES_FILA:
            seccion_actual = SECCIONES_FILA[categoria_norm]
            continue

        registro = metadatos_base(anio, tabla, pregunta, nombre_ambito(categoria, seccion_actual))
        registro["categoria"] = categoria

        for col, nombre_columna in columnas_valor.items():
            valor = limpiar_texto(row.iloc[col]) if col < len(row) else ""
            registro[nombre_columna] = pd.to_numeric(valor, errors="coerce")

        registros.append(registro)

    columnas_base = ["anio", "tabla", "pregunta", "ambito", "categoria"]
    columnas_valores = [nombre for _, nombre in sorted(columnas_valor.items())]
    return pd.DataFrame(registros)[columnas_base + columnas_valores]


## EAL-17 en formato ancho


In [10]:

# ============================================================
# EAL-17 A FORMATO ANCHO
# ============================================================

df_17 = process_eal17_18_wide("EAL-17", anio=ANIO)

print("df_17:", df_17.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_17)


df_17: (32, 11)


,anio,tabla,pregunta,ambito,categoria,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - IMPARTICIÓN DE CURSOS Y OTROS TIPOS DE FORMACIÓN (1),EMPRESAS QUE PROPORCIONAN FORMACIÓN - SÓLO IMPARTICIÓN DE CURSOS (1),EMPRESAS QUE PROPORCIONAN FORMACIÓN - SÓLO OTROS TIPOS DE FORMACIÓN (1),EMPRESAS QUE NO PROPORCIONAN FORMACIÓN
0,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",total_empresas,TOTAL,100,73.468,60.220,25.366,14.415,26.532
1,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 5 a 9 trabajadores,100,65.319,51.321,28.014,20.665,34.681
2,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 10 a 49 trabajadores,100,79.228,64.481,25.163,10.356,20.772
3,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 50 a 249 trabajadores,100,94.759,79.559,15.091,5.349,5.241
4,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 250 a 499 trabajadores,100,98.371,85.856,11.710,2.434,1.629
5,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,Más de 499 trabajadores,100,99.298,88.305,9.742,1.953,0.702
6,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Industria,100,76.222,64.223,20.911,14.866,23.778
7,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Construcción,100,77.713,61.593,29.467,8.940,22.287
8,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Comercio y reparación de vehículos,100,73.959,62.349,23.788,13.863,26.041
9,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Transporte y almacenamiento,100,77.405,55.376,34.796,9.828,22.595


## EAL-18 en formato ancho


In [11]:

# ============================================================
# EAL-18 A FORMATO ANCHO
# ============================================================

df_18 = process_eal17_18_wide("EAL-18", anio=ANIO)

print("df_18:", df_18.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_18)


df_18: (32, 11)


,anio,tabla,pregunta,ambito,categoria,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - PROPORCIONARON FORMACIÓN (1),DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - NO PROPORCIONARON FORMACIÓN (1),NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - PROPORCIONARON FORMACIÓN (2),NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - NO PROPORCIONARON FORMACIÓN (2)
0,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,total_empresas,TOTAL,29.123694,89.974511,10.025489,70.876306,66.570590,33.429410
1,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,tamano_empresa,De 5 a 9 trabajadores,23.671862,83.544726,16.455274,76.328138,59.481181,40.518819
2,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,tamano_empresa,De 10 a 49 trabajadores,31.724690,93.100741,6.899259,68.275310,72.749236,27.250764
3,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,tamano_empresa,De 50 a 249 trabajadores,48.701094,98.742720,1.266525,51.298906,90.863612,9.136388
4,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,tamano_empresa,De 250 a 499 trabajadores,58.708189,100.000000,0.000000,41.253364,96.831314,3.168686
5,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,tamano_empresa,Más de 499 trabajadores,62.753623,99.769053,0.230947,37.246377,98.702983,1.297017
6,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,sector,Industria,29.340514,91.943067,8.063373,70.659486,69.761720,30.238280
7,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,sector,Construcción,27.053957,90.611641,9.388359,72.946043,73.016647,26.983353
8,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,sector,Comercio y reparación de vehículos,28.467349,90.868950,9.131050,71.532651,67.384332,32.615668
9,2024,EAL-18,EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO DE LA E...,sector,Transporte y almacenamiento,25.901905,95.031299,4.968701,74.098095,71.218545,28.781455


## EAL-18a, EAL-18b, EAL-18c, EAL-19 y EAL-20 en formato ancho

Estas hojas tienen una estructura más simple: una columna de categoría y varias columnas de valores. Se procesan con una función común.


In [12]:

# ============================================================
# FUNCIONES PARA FORMATO ANCHO SIMPLE
# ============================================================

MARCADORES_SECCION = {
    "TAMAÑO DE LA EMPRESA",
    "ACTIVIDAD ECONÓMICA",
    "COMUNIDAD AUTÓNOMA",
}


def process_hoja_ancha_simple(hoja, fila_cabecera, fila_inicio_datos, anio=2024):
    df = leer_csv_eal(hoja)
    titulo = detectar_titulo(df, hoja)
    tabla, pregunta = extraer_tabla_pregunta(titulo)
    columnas_valor = hacer_columnas_unicas(df.iloc[fila_cabecera, 1:].tolist())

    registros = []
    seccion_actual = None
    for _, row in df.iloc[fila_inicio_datos:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "" or categoria.startswith("("):
            continue

        categoria_norm = categoria.upper()
        if categoria_norm in SECCIONES_FILA:
            seccion_actual = SECCIONES_FILA[categoria_norm]
            continue
        if categoria_norm in MARCADORES_SECCION:
            continue

        registro = metadatos_base(anio, tabla, pregunta, nombre_ambito(categoria, seccion_actual))
        registro["categoria"] = categoria

        for offset, columna in enumerate(columnas_valor, start=1):
            valor = limpiar_texto(row.iloc[offset]) if offset < len(row) else ""
            registro[columna] = pd.to_numeric(valor, errors="coerce")

        if all(pd.isna(registro[col]) for col in columnas_valor):
            continue

        registros.append(registro)

    columnas_base = ["anio", "tabla", "pregunta", "ambito", "categoria"]
    return pd.DataFrame(registros)[columnas_base + columnas_valor]


## EAL-18a en formato ancho


In [13]:
# ============================================================
# EAL-18A EN FORMATO ANCHO
# ============================================================

df_18a = process_hoja_ancha_simple("EAL-18a", fila_cabecera=4, fila_inicio_datos=5, anio=ANIO)

print("df_18a:", df_18a.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_18a)


df_18a: (32, 11)


,anio,tabla,pregunta,ambito,categoria,TOTAL,CURSOS DE FORMACIÓN DISEÑADOS Y GESTIONADOS POR SU EMPRESA,CURSOS DE FORMACIÓN DISEÑADOS Y GESTIONADOS POR OTRA ORGANIZACIÓN,"FORMACIÓN PLANIFICADA EN EL PUESTO DE TRABAJO, UTILIZANDO LOS MEDIOS HABITUALES DE TRABAJO","APRENDIZAJE PLANIFICADO A PARTIR DE ROTACIÓN DE PUESTOS DE TRABAJO, INTERCAMBIOS, ETC.","PARTICIPACIÓN EN CONFERENCIAS, SEMINARIOS, GRUPOS DE TRABAJO, TALLERES O FERIAS DE MUESTRAS"
0,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",total_empresas,TOTAL,73.468278,34.269329,73.709969,56.005492,27.375278,35.086542
1,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",tamano_empresa,De 5 a 9 trabajadores,65.319423,27.217848,65.989117,51.141371,22.937813,32.188656
2,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",tamano_empresa,De 10 a 49 trabajadores,79.227905,35.028637,79.002872,57.331022,29.277350,33.998708
3,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",tamano_empresa,De 50 a 249 trabajadores,94.759353,58.777973,84.135506,70.508861,36.475507,51.399249
4,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",tamano_empresa,De 250 a 499 trabajadores,98.385236,76.162564,85.384916,76.865963,46.697929,57.796014
5,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",tamano_empresa,Más de 499 trabajadores,99.275362,85.450122,84.963504,80.048662,51.532847,65.596107
6,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",sector,Industria,76.222600,34.293577,76.121179,61.910901,36.081017,33.867169
7,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",sector,Construcción,77.712290,22.420652,84.768610,57.771672,21.273003,23.030251
8,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",sector,Comercio y reparación de vehículos,73.958727,38.680615,71.445779,54.400700,29.940268,41.624315
9,2024,EAL-18a,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN MEDIO UTILIZADO POR TAMAÑO DE LA EMPRESA, ACTIVIDAD E...",sector,Transporte y almacenamiento,77.406769,30.568829,79.668783,52.052104,24.258690,20.416312


## EAL-18b en formato ancho


In [14]:
# ============================================================
# EAL-18B EN FORMATO ANCHO
# ============================================================

df_18b = process_hoja_ancha_simple("EAL-18b", fila_cabecera=5, fila_inicio_datos=7, anio=ANIO)

print("df_18b:", df_18b.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_18b)


df_18b: (31, 13)


,anio,tabla,pregunta,ambito,categoria,EMPRESAS QUE REALIZARON SUSPENSIÓN DE CONTRATO/REDUCCIÓN DE JORNADA,TOTAL,EL 0'%,MÁS DEL 0% HASTA EL 25%,MÁS DEL 25% HASTA EL 50%,MÁS DEL 50% HASTA EL 75%,MÁS DEL 75% HASTA EL 100%,EL 100%
0,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,tamano_empresa,De 5 a 9 trabajadores,0.763845,100,85.635,9.391,1.418,2.351,0.000,1.206
1,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,tamano_empresa,De 10 a 49 trabajadores,1.563983,100,82.378,13.232,0.000,2.052,0.312,2.025
2,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,tamano_empresa,De 50 a 249 trabajadores,3.367701,100,64.167,23.950,4.927,1.052,1.926,3.978
3,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,tamano_empresa,De 250 a 499 trabajadores,9.919262,100,69.215,19.402,4.646,0.000,3.834,2.904
4,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,tamano_empresa,Más de 499 trabajadores,11.787440,100,66.136,26.049,5.791,1.364,0.325,0.334
5,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,sector,Industria,3.526077,100,79.042,12.156,0.385,1.863,0.546,6.008
6,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,sector,Construcción,0.131222,100,84.067,8.563,0.058,5.920,1.391,0.000
7,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,sector,Comercio y reparación de vehículos,0.230648,100,85.276,11.029,1.964,0.000,0.000,1.732
8,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,sector,Transporte y almacenamiento,0.851236,100,82.853,16.628,0.328,0.000,0.191,0.000
9,2024,EAL-18b,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES EN SUSPENSIÓN DE CONTRATO Y/O REDUCCION DE JORNADA POR TAMA...,sector,Hostelería,2.805053,100,86.534,13.319,0.128,0.019,0.000,0.000


## EAL-18c en formato ancho


In [15]:
# ============================================================
# EAL-18C EN FORMATO ANCHO
# ============================================================

df_18c = process_hoja_ancha_simple("EAL-18c", fila_cabecera=4, fila_inicio_datos=6, anio=ANIO)

print("df_18c:", df_18c.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_18c)


df_18c: (31, 6)


,anio,tabla,pregunta,ambito,categoria,PORCENTAJE DE EMPRESAS QUE PROPORCIONARON FORMACIÓN EN SEGURIDAD Y SALUD LABORAL
0,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",tamano_empresa,De 5 a 9 trabajadores,68.611
1,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",tamano_empresa,De 10 a 49 trabajadores,81.343
2,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",tamano_empresa,De 50 a 249 trabajadores,92.156
3,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",tamano_empresa,De 250 a 499 trabajadores,95.577
4,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",tamano_empresa,Más de 499 trabajadores,96.501
5,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",sector,Industria,81.214
6,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",sector,Construcción,81.458
7,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",sector,Comercio y reparación de vehículos,74.164
8,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",sector,Transporte y almacenamiento,81.207
9,2024,EAL-18c,"EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SOBRE SEGURIDAD, SALUD E HIGIENE EN EL TRABAJO POR TAMAÑO D...",sector,Hostelería,73.552


## EAL-19 en formato ancho


In [16]:
# ============================================================
# EAL-19 EN FORMATO ANCHO
# ============================================================

df_19 = process_hoja_ancha_simple("EAL-19", fila_cabecera=4, fila_inicio_datos=5, anio=ANIO)

print("df_19:", df_19.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_19)


df_19: (6, 9)


,anio,tabla,pregunta,ambito,categoria,TOTAL,INDUSTRIA,CONSTRUCCIÓN,SERVICIOS
0,2024,EAL-19,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,total_empresas,TOTAL,73.468278,76.222600,77.712290,72.148643
1,2024,EAL-19,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,total_empresas,De 5 a 9 trabajadores,65.319423,63.058146,71.260095,64.621839
2,2024,EAL-19,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,total_empresas,De 10 a 49 trabajadores,79.227905,80.668769,83.545151,78.058545
3,2024,EAL-19,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,total_empresas,De 50 a 249 trabajadores,94.759353,97.169168,95.046440,93.740024
4,2024,EAL-19,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,total_empresas,De 250 a 499 trabajadores,98.385236,99.467377,100.000000,97.818599
5,2024,EAL-19,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,total_empresas,Más de 499 trabajadores,99.275362,99.782609,98.717949,99.151436


## EAL-20 en formato ancho


In [17]:
# ============================================================
# EAL-20 EN FORMATO ANCHO
# ============================================================

df_20 = process_hoja_ancha_simple("EAL-20", fila_cabecera=4, fila_inicio_datos=5, anio=ANIO)

print("df_20:", df_20.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_20)


df_20: (11, 11)


,anio,tabla,pregunta,ambito,categoria,TOTAL,DE 5 A 9 TRABAJADORES,DE 10 A 49 TRABAJADORES,DE 50 A 249 TRABAJADORES,DE 250 A 499 TRABAJADORES,MÁS DE 499 TRABAJADORES
0,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,De dirección,16.347340,15.343619,15.804993,22.430750,25.322392,30.510949
1,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,De trabajo en equipo,28.494848,29.855895,26.931117,28.578895,34.153966,33.333333
2,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,De atención al público/ trato a clientes,24.530267,27.373940,22.586210,19.846059,21.570926,27.055961
3,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,Administrativas de oficina,17.027644,15.670464,18.738422,15.816981,12.817507,13.333333
4,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,Técnicas específicas del puesto de trabajo,50.838810,48.254094,52.131239,56.943983,55.763970,54.014599
5,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,"De resolución de problemas (localización de problemas o fallos, análisis de sus causas y búsqueda de soluciones)",11.209969,11.767293,10.401264,11.911436,15.787417,13.187348
6,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,En lenguas extranjeras,7.478712,3.941978,7.740614,21.100394,27.471669,27.639903
7,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,Generales de tecnologías de la información,12.666991,11.324888,12.758545,17.261367,21.492771,25.109489
8,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,Profesionales de tecnologías de la información,5.310919,3.812620,5.996873,8.690075,7.854631,12.749392
9,2024,EAL-20,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS QUE SE H...,total_empresas,Básicas de cálculo y/o comunicación oral o escrita,1.254936,1.059013,1.173541,2.537179,2.110199,2.822384


## Unión de hojas compatibles

Se unen solo las hojas con una estructura de filas similar (`EAL-17`, `EAL-18`, `EAL-18a`, `EAL-18b`, `EAL-18c`). `EAL-16`, `EAL-19` y `EAL-20` se mantienen separadas porque responden a otra lógica de tabla.


In [18]:

# ============================================================
# UNIR HOJAS COMPATIBLES EN FORMATO ANCHO
# ============================================================

hojas_compatibles_ancho = [df_17, df_18, df_18a, df_18b, df_18c]

df_compatibles_ancho = pd.concat(
    hojas_compatibles_ancho,
    ignore_index=True,
    sort=False,
)

print("df_compatibles_ancho:", df_compatibles_ancho.shape)
print("Tablas incluidas:", df_compatibles_ancho["tabla"].unique().tolist())

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_compatibles_ancho)


df_compatibles_ancho: (158, 30)
Tablas incluidas: ['EAL-17', 'EAL-18', 'EAL-18a', 'EAL-18b', 'EAL-18c']


,anio,tabla,pregunta,ambito,categoria,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - IMPARTICIÓN DE CURSOS Y OTROS TIPOS DE FORMACIÓN (1),EMPRESAS QUE PROPORCIONAN FORMACIÓN - SÓLO IMPARTICIÓN DE CURSOS (1),EMPRESAS QUE PROPORCIONAN FORMACIÓN - SÓLO OTROS TIPOS DE FORMACIÓN (1),EMPRESAS QUE NO PROPORCIONAN FORMACIÓN,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - PROPORCIONARON FORMACIÓN (1),DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - NO PROPORCIONARON FORMACIÓN (1),NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - PROPORCIONARON FORMACIÓN (2),NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - NO PROPORCIONARON FORMACIÓN (2),CURSOS DE FORMACIÓN DISEÑADOS Y GESTIONADOS POR SU EMPRESA,CURSOS DE FORMACIÓN DISEÑADOS Y GESTIONADOS POR OTRA ORGANIZACIÓN,"FORMACIÓN PLANIFICADA EN EL PUESTO DE TRABAJO, UTILIZANDO LOS MEDIOS HABITUALES DE TRABAJO","APRENDIZAJE PLANIFICADO A PARTIR DE ROTACIÓN DE PUESTOS DE TRABAJO, INTERCAMBIOS, ETC.","PARTICIPACIÓN EN CONFERENCIAS, SEMINARIOS, GRUPOS DE TRABAJO, TALLERES O FERIAS DE MUESTRAS",EMPRESAS QUE REALIZARON SUSPENSIÓN DE CONTRATO/REDUCCIÓN DE JORNADA,EL 0'%,MÁS DEL 0% HASTA EL 25%,MÁS DEL 25% HASTA EL 50%,MÁS DEL 50% HASTA EL 75%,MÁS DEL 75% HASTA EL 100%,EL 100%,PORCENTAJE DE EMPRESAS QUE PROPORCIONARON FORMACIÓN EN SEGURIDAD Y SALUD LABORAL
0,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",total_empresas,TOTAL,100.000000,73.468,60.220,25.366,14.415,26.532,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 5 a 9 trabajadores,100.000000,65.319,51.321,28.014,20.665,34.681,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 10 a 49 trabajadores,100.000000,79.228,64.481,25.163,10.356,20.772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 50 a 249 trabajadores,100.000000,94.759,79.559,15.091,5.349,5.241,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 250 a 499 trabajadores,100.000000,98.371,85.856,11.710,2.434,1.629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,Más de 499 trabajadores,100.000000,99.298,88.305,9.742,1.953,0.702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Industria,100.000000,76.222,64.223,20.911,14.866,23.778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Construcción,100.000000,77.713,61.593,29.467,8.940,22.287,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Comercio y reparación de vehículos,100.000000,73.959,62.349,23.788,13.863,26.041,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2024,EAL-17,"EMPRESAS SEGÚN SI I

## Versión larga de las hojas compatibles

Se genera una vista larga para análisis posterior, sin guardar archivos. Las columnas quedan alineadas con Sabina: metadatos + `variable` + `porcentaje`.


In [19]:

# ============================================================
# PASAR HOJAS COMPATIBLES A FORMATO LARGO
# ============================================================

columnas_id = ["anio", "tabla", "pregunta", "ambito", "categoria"]
columnas_valor = [columna for columna in df_compatibles_ancho.columns if columna not in columnas_id]

df_compatibles_largo = to_long_format(
    df=df_compatibles_ancho,
    id_vars=columnas_id,
    value_vars=columnas_valor,
    var_name="variable",
    value_name="porcentaje",
)

print("df_compatibles_largo:", df_compatibles_largo.shape)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_compatibles_largo)


df_compatibles_largo: (855, 7)


,anio,tabla,pregunta,ambito,categoria,variable,porcentaje
0,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",total_empresas,TOTAL,TOTAL,100.000000
1,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 5 a 9 trabajadores,TOTAL,100.000000
2,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 10 a 49 trabajadores,TOTAL,100.000000
3,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 50 a 249 trabajadores,TOTAL,100.000000
4,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,De 250 a 499 trabajadores,TOTAL,100.000000
5,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",tamano_empresa,Más de 499 trabajadores,TOTAL,100.000000
6,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Industria,TOTAL,100.000000
7,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Construcción,TOTAL,100.000000
8,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Comercio y reparación de vehículos,TOTAL,100.000000
9,2024,EAL-17,"EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",sector,Transporte y almacenamiento,TOTAL,100.000000
